In [ ]:
pip install wtpsplit #sentence-transformers numpy

In [54]:
import json
import re
from functools import lru_cache
from pathlib import Path
from typing import List
import numpy as np
from scipy.signal import find_peaks
from sentence_transformers import SentenceTransformer

@lru_cache(maxsize=1)
def _get_embedding_model(model_name: str = "BAAI/bge-m3"):
    """Loads the sentence-transformer model once per process"""
    return SentenceTransformer(model_name)

def embed_texts(texts: List[str]) -> np.ndarray:
    """Generates embeddings for a list of texts using a sentence-transformer model."""
    model = _get_embedding_model()
    return model.encode(texts, convert_to_numpy=True)

In [55]:
#all the section in ECHR documents
SECTION_HEADERS = [
    "PROCEDURE",
    "THE FACTS",
    "AS TO THE FACTS",
    "COMPLAINTS",
    "PROCEEDINGS BEFORE THE COMMISSION",
    "THE LAW",
    "AS TO THE LAW",
]

#sections to keep
KEEP_SECTIONS = {"COMPLAINTS", "THE LAW", "AS TO THE LAW"}

DISSENTING_OPINION_RE = re.compile(
    r"^DISSENTING OPINION OF (?:JUDGE|JUDGES)\s+[A-ZÀ-Ü.\s]+$", re.MULTILINE
)

In [121]:
text_14 = """PROCEDURE
1.   The case was referred to the Court by the European Commission of Human Rights ("the Commission") on 29 May 1995, and by the Government of the Swiss Confederation ("the Government") on 26 June 1995, within the three-month period laid down by Article 32 para. 1 and Article 47 (art. 32-1, art. 47) of the Convention.  It originated in an application (no. 23218/94) against Switzerland lodged with the Commission under Article 25 (art. 25) by a Turkish national, Mr Riza Gül, on 31 December 1993.
The Commission’s request referred to Articles 44 and 48 (art. 44, art. 48) and to the declaration whereby Switzerland recognised the compulsory jurisdiction of the Court (Article 46) (art. 46); the Government’s application referred to Articles 45, 47 and 48 (art. 45, art. 47, art. 48).  The object of the request and of the application was to obtain a decision as to whether the facts of the case disclosed a breach by the respondent State of its obligations under Article 8 (art. 8) of the Convention.
2.   In response to the enquiry made in accordance with Rule 33 para. 3 (d) of Rules of Court A, the applicant stated that he wished to take part in the proceedings and designated the lawyers who would represent him (Rule 30).  The Turkish Government, having been informed by the Registrar of their right to intervene in the proceedings (Article 48 (b) of the Convention and Rule 33 para. 3 (b)) (art. 48-b), did not indicate any intention of so doing.
3.   The Chamber to be constituted included ex officio Mr L. Wildhaber, the elected judge of Swiss nationality (Article 43 of the Convention) (art. 43), and Mr R. Bernhardt, the Vice-President of the Court (Rule 21 para. 4).  On 8 June 1995, in the presence of the Registrar, the President of the Court drew by lot the names of the other seven members, namely Mr F. Matscher, Mr C. Russo, Mr N. Valticos, Mr S.K. Martens, Mrs E. Palm, Mr M.A. Lopes Rocha and Mr K. Jungwiert (Article 43 in fine of the Convention and Rule 21 para. 5) (art. 43).
4.   As President of the Chamber (Rule 21 para. 6), Mr Bernhardt, acting through the Registrar, consulted the Agent of the Government, the applicant’s lawyers and the Delegate of the Commission on the organisation of the proceedings (Rules 37 para. 1 and 38).  Pursuant to the order made in consequence, the Registrar received the memorials of the applicant and the Government on 3 and 11 August 1995 respectively.  On 4 September 1995 the Secretary to the Commission informed the Registrar that the Delegate would submit his observations at the hearing.
On 25 August 1995 the Commission produced the file on the proceedings before it, as requested by the Registrar on the President’s instructions.
5.   In accordance with the President’s decision, the hearing took place in public in the Human Rights Building, Strasbourg, on 25 October 1995.  The Court had held a preparatory meeting beforehand.
There appeared before the Court:
(a) for the Government
		Mr O. JACOT-GUILLARMOD, Assistant Director,
			Head of the International Affairs Division,
			Federal Office of Justice,	Agent,
		Mr F. SCHÜRMANN, Deputy Head of the European
			Law and International Affairs Section,
			Federal Office of Justice,
		Mrs S. MARCONATO, Legal Officer,
			Federal Aliens Office,	Counsel;
(b) for the Commission
		Mr H. DANELIUS,	Delegate;
(c) for the applicant
		Mr R. PLENDER QC,
		Mr J. WALKER, Fürsprech,	Counsel.
The Court heard addresses by Mr Danelius, Mr Plender and Mr Jacot-Guillarmod, and the latter’s reply to the question asked by one member of the Court.
On 25 and 27 October 1995 the Registrar received the Government’s and the applicant’s written replies to that question.
AS TO THE FACTS
I.   CIRCUMSTANCES OF THE CASE
A. Situation of the applicant and part of his family in Switzerland
6.   Mr Gül is a Turkish national, who was born in 1947 and now lives with his wife at Pratteln in the canton of Basle Rural, Switzerland.
7.   Until 1983 he lived with his wife and their two sons, Tuncay (born on 12 October 1971) and Ersin (born on 20 January 1983), in the town of Gümüshane in Turkey.  On 25 April 1983 he travelled to Switzerland, where he applied for political asylum as a Kurd and former member of the Turkish Social Democratic Party ("the CHP").  He worked in a restaurant there until 1990, when he fell ill.  Since then he has been in receipt of a partial-invalidity pension.
8.   In 1987 the applicant’s wife, who had remained in Turkey with their two sons, seriously burned herself during a fit brought on by her epilepsy, from which she had suffered since 1982.  In December 1987, having found that it was impossible for her to obtain proper treatment in the area where she was then living, she joined her husband in Switzerland, where she was taken into hospital as an emergency case.  Two of the fingers of her left hand were amputated.
9.   On 19 September 1988 in Switzerland Mrs Gül gave birth to her third child, Nursal, a daughter.  As she still suffered from epilepsy, she could not take care of the baby, who was placed in a home in Switzerland, where she has remained ever since.  In a written declaration dated 31 March 1989, a Pratteln specialist in internal medicine stated that a return to Turkey would be impossible for Mrs Gül and might even prove fatal to her, given her serious medical condition.
10.   On 9 February 1989 the Minister for Refugees rejected Mr Gül’s application for political asylum, on the ground that he had not been able to establish that he personally had been a victim of persecution, as the general situation of the Kurdish population in Turkey was not in itself sufficient to justify granting political asylum.  He went on to say that, according to reliable sources, no measures were being taken by the State authorities against former members of the CHP, and ordered the applicant to leave Switzerland by 30 April 1989, failing which he would be deported.
On 10 March 1989 the applicant appealed against the above decision to the Federal Justice and Police Department.  He asserted that the collective repression of Kurds in Turkey, of which he himself had been a victim, in itself justified granting political asylum.  In addition, at the time when he had fled to Switzerland all political parties had been proscribed and their members - especially the members of left-wing parties like the CHP - were being prosecuted.  He could not therefore be required to return to Turkey, and this would be in breach of Article 3 (art. 3) of the Convention.
11.   In a letter of 26 June 1989, the Basle Rural Cantonal Aliens Police (Fremdenpolizei) informed the applicant’s lawyer that they supported Mr Gül’s request for a residence permit (Aufenthaltsbewilligung) on humanitarian grounds in respect of himself, his wife and his daughter Nursal.
In view of the length of time Mr Gül had been living in Switzerland and his wife’s precarious state of health, the police considered that the conditions for the issue of such a permit laid down in Article 13 (f) of the Federal Council’s Order Limiting the Number of Aliens ("the OLNA" - see paragraph 21 below) had been satisfied.  The final decision to grant a residence permit was given by the Federal Aliens Office on 15 February 1990.
12.   As the Federal Justice and Police Department had informed Mr Gül that his application for political asylum had only very limited prospects of success on appeal, he withdrew it.  The authorities took formal note of this on 8 November 1989.
B. Steps taken by the applicant with a view to bringing his two sons to Switzerland
1. Before the Basle Rural Cantonal Aliens Police
13.   On 14 May 1990 Mr Gül asked the Basle Rural Cantonal Aliens Police for permission to bring to Switzerland his two sons, Tuncay and Ersin, who had remained in Turkey.
14.   In a decision of 19 September 1990 the Aliens Police rejected Mr Gül’s request, on the ground that the conditions for family reunion had not been satisfied (Article 39 of the OLNA - see paragraph 21 below).  Firstly, the Gül family’s flat did not conform to the standards laid down and, secondly, the applicant did not have sufficient means to provide for his family.  In any event, Tuncay was already eighteen and was therefore ineligible for a residence permit under the rules governing family reunion.
2. Before the Basle Rural cantonal government
15.   On 1 October 1990 the applicant appealed against this decision to the Basle Rural cantonal government (Regierungsrat). He argued that the residence permit issued to him and his wife under Article 13 (f) of the OLNA should have been extended to include his two sons, as his personal circumstances made it an extremely serious case.  Since it was impossible to return to Turkey because of his wife’s precarious state of health and the length of time he had lived abroad, the family could be brought back together only in Switzerland.  Both Article 8 (art. 8) of the European Convention on Human Rights, guaranteeing the right to respect for family life, and the United Nations Convention on the Rights of the Child gave the two boys the right to join their parents in Switzerland.  If the cantonal government were nevertheless to rely on the provisions of Articles 38 et seq. of the OLNA (see paragraph 21 below) on family reunion, the younger son, Ersin, could and should be permitted to exercise that right. There was enough room for him in the family’s flat and Mr Gül’s financial resources were sufficient to provide for his family.
16.   On 30 July 1991 the Basle Rural cantonal government dismissed the applicant’s appeal.  It pointed out that under section 4 of the Federal Residence and Settlement of Aliens Act ("the RSAA" - see paragraph 20 below) the question whether to grant a residence permit (Aufenthaltsbewilligung) or settlement permit (Niederlassungsbewilligung) was determined by the competent cantonal authorities with unfettered discretion (nach freiem Ermessen), having regard to the relevant statutory provisions and international agreements.  In that connection, the authorities had to take account of the country’s moral and economic interests, and of the degree of immigrant penetration (Überfremdung).
The cantonal government then considered whether Mr Gül’s two sons could rely on a right to obtain permission to reside in Switzerland (Anwesenheitsbewilligung) on the basis of the statutory provisions, as the agreement on settlement concluded by Turkey and Switzerland on 13 December 1990 did not confer such a right.
Under section 17 (2) of the RSAA (see paragraph 20 below) a minor did not have such a right unless his parent was in possession of a settlement permit.  As Mr and Mrs Gül only had a residence permit, they could not rely on that provision in order to assert a right to family reunion.  As for the guarantees set forth in Article 8 (art. 8) of the Convention, only Swiss nationals or persons in possession of a settlement permit could rely on these; Mr and Mrs Gül fell into neither of those categories.
Articles 38 et seq. of the OLNA (see paragraph 21 below) did not confer a right but merely set out the minimum conditions to be satisfied before family reunion could be authorised.  The cantonal authorities had the final say on the matter, and in reaching their decision they had unfettered discretion.  It being established that the provisions concerned could only apply, if at all, to the minor son, Ersin, the cantonal government listed the minimum conditions which Article 39 para. 1 of the OLNA (see paragraph 21 below) required to be satisfied by a foreigner living in Switzerland before family reunion could be authorised, namely:
(a) his residence and, where relevant, his gainful employment should appear to be sufficiently stable;
(b) he should live with his family and occupy accommodation suitable for that purpose;
(c) he should have sufficient means to support his family;
and
(d) firm arrangements should have been made for the care of any children who still needed their parents’ presence.
The cantonal government did not determine points (a) and (b), but carefully considered points (c) and (d), on which it gave the following decision:
"(c) The calculations made by the Aliens Police and the cantonal government’s legal service, which investigated the case, show that Mr Gül has not satisfied the condition laid down in Article 39 para. 1 (c) of the OLNA.  He does not have sufficient means to support his family during their residence in Switzerland.  According to the reference calculation, Mr Gül should have a monthly net income of at least 2,710 Swiss francs (CHF) if he is not to fall below the minimum standard of living.  That figure is derived from the base rates used by the cantonal social security office for assessing the likelihood of reliance on social security, which are on the whole identical with the base rates adopted by the Swiss Conference on Public Assistance for the calculation of financial support.  These base rates are used to establish the monthly living expenses of the foreigner concerned and the members of his family seeking to join him, which have to be covered by his income.  This must be sufficient to provide not only the basic necessities of life but also a minimum standard of living. In this way the legitimate interest of the public authorities in preventing the family from becoming a burden on the social security services is also taken into account.
Mr Gül’s net monthly income is CHF 2,060, which falls CHF 650 short of the amount required for the minimum standard of living as calculated by the social security services.  The cost of keeping the youngest child, Nursal, in a children’s home has not been taken into account, as it is not known who pays for this.  The calculation of income is based on pay-slips from 1989, the latest available.  On 23 October 1990 the Liestal Cantonal Hospital sent the Basle Rural Aliens Police a medical certificate stating that Mr Gül was 100% unfit for work and would remain so for a period that it was not possible to determine.  On enquiry being made, it was confirmed in a medical certificate dated 19 April 1991 that Mr Gül had suffered from 100% incapacity since April 1990 and would remain unfit for work for the foreseeable future.  The Pratteln municipal social security services stated in a letter of 11 June 1991 that Mr Gül would have to have several operations and that for the time being he was waiting to be awarded an invalidity pension. For the first three months of this year alone the social security services have paid the Gül family CHF 8,731.75, and the family will remain dependent on social security payments.  In June 1991 Mr Gül stated during a personal interview with the subordinate authority (Vorinstanz) that his family was at that time entirely dependent on social security payments.  He therefore has no other source of income.  At present the social security services are paying the Gül family the amount needed by a three-person family, but no more.  The social security services cannot be expected to provide for children arriving from abroad when it is known in advance that they will have to support them. Nor can Mr Gül support his other children from his own resources.  For that reason alone the application for family reunion must be refused.
(d) Article 39 para. 1 (d) also requires firm arrangements to be made for the care of children.  But Mrs Gül, for reasons connected with her illness, is not mentally or physically capable of keeping her daughter Nursal with her and looking after her.  That is why Nursal has been brought up in the "Auf Berg" children’s home in Seltisberg, where she is to remain.  It follows that if Mr and Mrs Gül’s eight-year-old son Ersin joined the family, it is not at all certain that firm arrangements could be made for his care.  He too would presumably have to be brought up in a children’s home, which is not the aim of family reunion. A medical certificate dated 18 April 1991 states that Mrs Gül is suffering from a serious illness which makes it necessary for her to have constant medical supervision and treatment.  She might even need to go into hospital again. That prospect makes it impossible to consider that firm arrangements have been made for the child’s care as the Order requires."
The cantonal government went on to say that residence permits issued on humanitarian grounds under Article 13 (f) of the OLNA could not in addition confer on the recipients a right to family reunion.  In order to ensure equal treatment for all aliens not having the right to reside in Switzerland, such reunion could take place only under Articles 38 et seq. of the OLNA.
Lastly, the cantonal government considered the situation of the younger boy from the standpoint of Article 36 of the OLNA (see paragraph 21 below), announcing its decision in the following terms:
"Ersin Gül is only eight.  It must be determined whether his entry into Switzerland would be in accordance with Article 36 of the OLNA, which requires an `important reason’ that is lacking in this case.  There is no special reason for treating Ersin Gül differently from other children wishing to rejoin their families in respect of whom the conditions laid down in Articles 38 et seq. of the OLNA have not been satisfied.  Another reason for refusing to admit him to Switzerland is the fact that Ersin and Tuncay Gül would be separated.  Ersin has lived with Tuncay since birth.  On the other hand, he has been separated from his father and mother for eight years and three and a half years respectively.  Having regard to the child’s welfare, which plays an important role in family reunion cases, the question arises, at the very least, whether it is reasonable to separate him from his brother and the environment he is used to in order to bring him to live with his mother, who is seriously ill and unable to keep him with her or look after him, and his father, who went away to Switzerland three month’s after Ersin’s birth, which means that he hardly knows him.  In view of all the circumstances, and having regard to the child’s welfare, the cantonal government considers that Ersin Gül should not be authorised to join his parents in Switzerland.  In any case, there is no important reason within the meaning of Article 36 of the OLNA which requires him to be admitted to Switzerland."
The cantonal government concluded that Mr Gül had not satisfied the conditions laid down for family reunion and that his children could not rely on Article 13 (f) or Article 36 of the OLNA either in order to come to Switzerland to join him.
3. In the Federal Court
17.   On 2 September 1991 the applicant lodged an administrative-law appeal with the Swiss Federal Court.  He repeated his previous arguments (see paragraph 15 above) and added that, because of the "special circumstances", Article 8 (art. 8) of the Convention gave his sons the right to obtain permission to reside in Switzerland.  The earlier issue of a residence permit on humanitarian grounds to himself, his wife and his daughter had been based on the finding that a return to Turkey was impossible, as it would put the health of his wife and daughter seriously at risk.  Mr Gül argued that the same considerations which had prevailed in the decision to grant that residence permit should prevent any withdrawal thereof, which would be tantamount to subjecting Mrs Gül, whose state of health was still causing concern, to inhuman and degrading treatment contrary to Article 3 (art. 3) of the Convention.  The residence permit issued to Mr and Mrs Gül on humanitarian grounds was therefore the equivalent of a settlement permit, and it followed that they had the right to family reunion, which could only take place in Switzerland.
18.   In a judgment of 2 July 1993 the Federal Court declared the applicant’s appeal inadmissible.  It pointed out that, pursuant to section 100 (b) (3) of the Federal Administration of Justice Act, an administrative-law appeal in an immigration-control case was inadmissible if it concerned the issue or refusal of permits to which federal legislation conferred no entitlement.  Like the cantonal government, the Federal Court found that neither section 17 (2) of the RSAA nor Article 8 (art. 8) of the Convention conferred such a right on an alien resident outside Switzerland whose parent living in Switzerland had only a residence permit, as Mr Gül did.  In particular, Article 8 (art. 8) of the Convention could be relied on only by a person who had the right of abode in Switzerland either by virtue of his Swiss nationality or by virtue of a settlement permit.  The court gave this ruling in the following terms:
"Article 8 (art. 8) of the European Convention on Human Rights guarantees the right to respect for family life.  In certain circumstances the right to be issued with a residence permit can be deduced from this (see ATF [Judgments of the Swiss Federal Court] 118 Ib 152 at 4, 157 at c; 116 Ib 355 at 1b; 109 Ib 185 at 2), so that Article 8 (art. 8) may be breached where an alien whose family lives in Switzerland is refused leave to enter the country. According to the Federal Court’s established case-law, however, a breach can occur only where the family members living in Switzerland themselves possess a well-established right of abode (Anwesenheitsrecht).  For that purpose, it is in principle necessary to have Swiss nationality or possess a settlement permit (see ATF 116 Ib 355 at 1b; 115 Ib 4 at 1d).  A mere residence permit is at any rate not sufficient unless it is based on a firmly established right (see ATF 111 Ib 163/4 at 1a), as the Federal Court has held in many unpublished judgments (most recently in the judgment of 6 April 1993 in the case of K., at 1b) ... That is, moreover, consistent with the new provisions on the legal status of aliens having family members in Switzerland (sections 7 and 17 (2) of the RSAA, as amended on 23 March 1990, which came into force on 1 January 1992). Under the Act the right to family reunion presupposes a firmly established right of abode, as pointed out above (at 1b).  Given that the legislature’s intention in adopting the amendment in question was precisely to take account of Article 8 (art. 8) of the European Convention on Human Rights (see Bbl [Federal Gazette] 1987 III, pp. 293 et seq., particularly pp. 321 and 322), there is no reason, when that provision (art. 8) of the Convention is invoked with regard to recognition of legal rights in the matter of residence permits, to go beyond what the Act itself expressly provides (see the Federal Court’s unpublished judgment of 6 April 1993 in the case of K., at 1b)."
The Federal Court also emphasised the differences between settlement permits and residence permits, stating:
"Unlike settlement permits, which are issued for an indefinite period (section 6 (1) of the Federal Residence and Settlement of Aliens Act - hereinafter the RSAA), residence permits are always subject to a time-limit (section 5 (1) of the RSAA).  Whatever the reason for granting the first residence permit, an alien must therefore allow for the possibility that his permit will not be renewed.  There could be many reasons for this, including, for example, police, economic or demographic considerations.  Although the alien’s personal circumstances have to be taken into account in the inquiry into the proportionality of the decision not to renew, that does not mean that the alien is on that account entitled to have his residence permit renewed.
The above statement of the law also applies to residence permits issued on humanitarian grounds.  The only effect of a finding that a case is an extremely serious one within the meaning of Article 13 (f) of the Federal Council’s Order Limiting the Number of Aliens of 6 October 1986 (hereinafter the OLNA - SR 823.21) is to exclude the alien concerned from the quotas laid down in that Order; it does not imply the existence of a right to a residence permit. The Aliens Police prefer to remain free to decide when such a permit should be issued (see ATF 119 Ib 35 at 1a).  In addition, the possibility cannot be ruled out that the particular circumstances which justified the issue of a residence permit on humanitarian grounds will subsequently cease to exist, or lose their significance to such an extent that not only will there no longer be any reason to exclude the person concerned from the quotas, but even renewal of the residence permit will no longer be justified.  Moreover, it is apparent from the rule established in Article 12 para. 2 of the OLNA that the conditions required for a finding that the case is an extremely serious one may subsequently cease to exist (see the unpublished judgment of 3 July 1992 in the case of P., at 6).  The question whether the case is of this type is therefore entirely separate from the question whether the person concerned has the right to obtain permission to reside in Switzerland by virtue of Article 8 (art. 8) of the European Convention on Human Rights (see ATF 115 Ib 8).
Furthermore, in the instant case the possibility cannot be entirely ruled out that in the future the medical or other reasons which led the authorities to grant the residence permit will lose their significance, or that new grounds justifying a refusal to renew the permit will become apparent.  The appellant can therefore not deduce from the fact that he is authorised to reside in Switzerland any right to the issue of a residence permit for his sons."
The Federal Court went on to say that the question how the OLNA should be applied to the issue of permits was not one it had to examine in connection with the administrative-law appeal, as the cantonal government had already looked into the question whether the Güls’ younger son could be issued with a residence permit under Article 36 of the OLNA.
C. Situation of the applicant’s son Ersin in Turkey
19.   Ersin has lived in Turkey since his birth, at first in Gümüshane until 1993 (with his mother until 1987), and then in Istanbul.
According to the Government, he is at present living, as is his grandfather, with the family of his elder brother Tuncay, and has been visited several times by his father.
The applicant maintained that Ersin frequently moved from one home to another and spent two or three days staying with various Kurdish families who used to live in the village where he was born, including the family of his elder brother.  Owing to his grandfather’s limited financial resources and the distance between the homes of some of these families and the school it was not possible for the boy to attend school on a regular basis.
As is evidenced by an article which appeared in the Turkish newspaper Sabah on 25 July 1995, Mr and Mrs Gül visited their son in Turkey in July and August 1995.
II.   RELEVANT DOMESTIC LAW
A. The Federal Residence and Settlement of Aliens Act  (RSAA) of 26 March 1931
20.   The Federal Residence and Settlement of Aliens Act provides:
Section 4
"The authority shall have discretion to decide, having regard to the relevant statutory provisions and treaties with foreign States, whether to grant residence or settlement permits."
Section 16
"1.   When deciding whether to grant a permit the authorities must take account of the country’s moral and economic interests, and of the degree of immigrant penetration.
..."
Section 17
"1.   As a general rule, the authority shall first issue only a residence permit, even if it is foreseen that the alien will establish his permanent residence in Switzerland.  In each case the Federal Aliens Office shall fix the date from which permission to settle may be granted.
2.   Where that date has already been fixed, or where the alien is in possession of a settlement permit, his spouse shall be entitled to a residence permit for as long as the couple continue to live together.  On completion of five years’ uninterrupted lawful residence the spouse shall also become entitled to a settlement permit.  Unmarried children under 18 shall have the right to be included in the settlement permit for as long as they continue to live with their parents.  These rights shall be extinguished if the beneficiary has engaged in conduct contrary to public policy."
Before 1 January 1992 the second paragraph of this section read:
"Where that date has already been fixed, or where the alien is in possession of a settlement permit, his wife and his children under 18 shall have the right to be included in the permit if they form part of his household."
B. The Order Limiting the Number of Aliens (OLNA) of 6 October 1986
21.   The relevant provisions of the Order Limiting the Number of Aliens are the following:
Article 13 - Exceptions
"The following categories of person shall not be included in the quotas:
...
(f) aliens issued with residence permits in extremely serious personal cases or on general policy grounds.
..."
Before 18 October 1989 the expression "extremely serious personal cases" read: "cases of extreme adversity".
Article 36 - Other aliens without gainful employment
"Residence permits may be issued to other aliens without gainful employment where important reasons so require."
Chapter 4: Family reunion
Article 38 - Principle
"1.   The Cantonal Aliens Police may authorise an alien to bring to Switzerland his spouse and his dependent unmarried children under 18.
..."
Article 39 - Conditions
"1.   An alien may be authorised to bring his family without being required to complete any qualifying period ...
(a) if his residence and, where relevant, his gainful employment appear to be sufficiently stable;
(b) if he lives with his family and occupies accommodation suitable for that purpose;
(c) if he has sufficient means to support his family; and
(d) if firm arrangements have been made for the care of any children who still need their parents’ presence.
2.   Accommodation is suitable if it meets the standards applicable to Swiss nationals in the area where the alien wishes to live."
Before 20 October 1993 the words "without being required to complete any qualifying period" were not part of the text.
C. Case-law of the Swiss Federal Court
22.   According to the established case-law of the Federal Court, a person is entitled under Article 8 (art. 8) of the Convention to join a member of his family in Switzerland if the latter is a Swiss national or is in possession of a settlement permit (Judgments of the Federal Court (ATF) vol. 116, part Ib, p. 355; vol. 115, part Ib, p. 4; vol. 111, part Ib, pp. 163 et seq.).
D. The convention on social security concluded by Switzerland and Turkey on 1 May 1969
23.   Replying to the question asked at the hearing by one member of the Court, the Government stated that by virtue of the convention on social security concluded by Switzerland and the Republic of Turkey on 1 May 1969, which came into force on 1 January 1972 with effect from 1 January 1969, invalidity insurance benefits payable in either country are also payable in the other.  In the instant case, if Mr Gül returned to Turkey, he would receive CHF 915, made up of his ordinary pension (CHF 436) and half of the supplementary pension paid in respect of his wife (CHF 131), his son Ersin (CHF 174) and his daughter Nursal (CHF 174).
The applicant asserted that only his invalidity pension, not the social security benefits, could be paid to him in Turkey. Moreover, his invalidity pension was currently under review; if his invalidity were to be assessed as less than 50%, his pension could no longer be transferred to Turkey.
PROCEEDINGS BEFORE THE COMMISSION
24.   Mr Gül applied to the Commission on 31 December 1993.  He alleged that the Swiss authorities’ refusal to allow his two sons, Tuncay and Ersin, to join him in Switzerland constituted a violation of Article 8 (art. 8) of the Convention.
25.   On 10 October 1994 the Commission declared the application (no. 23218/94) admissible as regards the complaint under Article 8 (art. 8) of the Convention concerning Ersin.  It declared the remainder of the application inadmissible.
In its report of 4 April 1995 (Article 31) (art. 31), it expressed the opinion, by fourteen votes to ten, that there had been a violation of Article 8 (art. 8).  The full text of the Commission’s opinion and of the two dissenting opinions contained in the report is reproduced as an annex to this judgment
FINAL SUBMISSIONS TO THE COURT
26.   In their memorial the Government asked the Court to hold in the instant case:
"primarily, that Article 8 (art. 8) of the Convention is not applicable; in the alternative, that there was no `interference’ by the Swiss public authorities with the applicant’s exercise of his right to enjoy a family life with his son Ersin;  in the further alternative, if such interference is held to have occurred, that it was justified under paragraph 2 of Article 8 (art. 8-2) of the Convention."
27.   The applicant asked the Court to find that the conditions laid down in Article 8 para. 2 (art. 8-2) of the Convention had not been satisfied and to uphold the Commission’s opinion on this point.
AS TO THE LAW
I.   ALLEGED VIOLATION OF ARTICLE 8 (art. 8) OF THE CONVENTION
28.   Mr Gül submitted that the Swiss authorities’ refusal to permit his son Ersin to join him in Switzerland had infringed his right to respect for his family life.  He relied on Article 8 (art. 8) of the Convention, which provides:
"1.   Everyone has the right to respect for his ... family life ...
2.   There shall be no interference by a public authority with the exercise of this right except such as is in accordance with the law and is necessary in a democratic society in the interests of national security, public safety or the economic well-being of the country, for the prevention of disorder or crime, for the protection of health or morals, or for the protection of the rights and freedoms of others."
29.   It is first necessary to determine whether there is a "family life" within the meaning of Article 8 (art. 8).
30.   The Government’s primary submission was that Article 8 (art. 8) was not applicable, since in the instant case the element of intention inherent in the concept of family life was missing.  Mr Gül had left Turkey when his younger son Ersin was three months old, and had never attempted to develop a family life in his country of origin.  In addition, the focus of that son’s family life was in Turkey since, even after his mother’s departure, the child had been taken in as a member of his elder brother’s family.  Furthermore, the fact that Mr and Mrs Gül’s daughter Nursal had been placed in a home in Switzerland showed that they were in any event incapable of assuming their parental responsibilities with regard to the boy.
31.   Like the applicant, the Commission considered that the bond between Mr Gül and his son Ersin amounted to "family life".
32.   The Court reiterates that it follows from the concept of family on which Article 8 (art. 8) is based that a child born of a marital union is ipso jure part of that relationship; hence, from the moment of the child’s birth and by the very fact of it, there exists between him and his parents a bond amounting to "family life" (see the Berrehab v. the Netherlands judgment of 21 June 1988, series A no. 138, p. 14, para. 21, and the Hokkanen v. Finland judgment of 23 September 1994, Series A no. 299-A, p. 19, para. 54) which subsequent events cannot break save in exceptional circumstances.
33.   Admittedly, Mr Gül left Turkey in 1983, when his son Ersin was only three months old (see paragraph 7 above); Mrs Gül left Ersin in 1987 because of her accident (see paragraph 8 above).
However, after obtaining a residence permit on humanitarian grounds in Switzerland in 1990, the applicant asked the Swiss authorities for permission to bring the boy, who was then six years old, to Switzerland (see paragraphs 11 and 13 above). Subsequently, he repeatedly asked the Swiss courts to allow his son to join him, before bringing his case before the Convention institutions.  Despite the distance, in geographical terms, between them, the applicant has made a number of visits to Turkey, the last of these being in July and August 1995 (see paragraph 19 above).  It cannot therefore be claimed that the bond of "family life" between them has been broken.
34.   Secondly, it is necessary to ascertain whether there was interference by the Swiss authorities with the applicant’s right under Article 8 (art. 8).
35.   Mr Gül submitted that the result in practice of the authorities’ persistent refusal to allow Ersin to join him in Switzerland had been to separate the family and make it impossible, owing to lack of sufficient financial resources, for the parents to maintain regular contacts with their son, whereas, according to the Court’s case-law, contacts between parents and child were of capital importance.  In addition, the length of time Mr Gül had lived in Switzerland, his invalidity and his wife’s ill-health made family reunion in Turkey an unrealistic prospect, so that the family could only be brought together again in Switzerland.
36.   The Government submitted that the applicant could not rely on a right to family reunion in Switzerland, as he had only a humanitarian permit, which was not a true settlement permit but merely a document authorising residence that could be withdrawn from him.  In addition, Switzerland had fully discharged the positive obligations arising under Article 8 para. 1 (art. 8-1), as the invalidity pension the applicant was in receipt of enabled him to make occasional visits to Turkey.  In any event, Switzerland was in no way responsible for the situation the Gül family was in.  Lastly, the Swiss authorities were not under any obligation to ensure that the applicant led an optimal family life in Switzerland.
37.   The Commission considered that where a parent wanted his minor child to live with him, preventing this amounted to interference with his right to respect for family life, and that the family would need to be reunited in Switzerland rather than in Turkey in view of Mr and Mrs Gül’s particular circumstances.
38.   The Court reiterates that the essential object of Article 8 (art. 8) is to protect the individual against arbitrary action by the public authorities.  There may in addition be positive obligations inherent in effective "respect" for family life. However, the boundaries between the State’s positive and negative obligations under this provision (art. 8) do not lend themselves to precise definition.  The applicable principles are, nonetheless, similar.  In both contexts regard must be had to the fair balance that has to be struck between the competing interests of the individual and of the community as a whole; and in both contexts the State enjoys a certain margin of appreciation (see, most recently, the Keegan v. Ireland judgment of 26 May 1994, Series A no. 290, p. 19, para. 49, and the Kroon and Others v. the Netherlands judgment of 27 October 1994, Series A no. 297-C, p. 56, para. 31).
The present case concerns not only family life but also immigration, and the extent of a State’s obligation to admit to its territory relatives of settled immigrants will vary according to the particular circumstances of the persons involved and the general interest.  As a matter of well-established international law and subject to its treaty obligations, a State has the right to control the entry of non-nationals into its territory (see, among other authorities, the Abdulaziz, Cabales and Balkandali v. the United Kingdom judgment of 28 May 1985, Series A no. 94, pp. 33-34, para. 67).
Moreover, where immigration is concerned, Article 8 (art. 8) cannot be considered to impose on a State a general obligation to respect the choice by married couples of the country of their matrimonial residence and to authorise family reunion in its territory.  In order to establish the scope of the State’s obligations, the facts of the case must be considered (see, mutatis mutandis, the Abdulaziz, Cabales and Balkandali judgment previously cited, p. 34, para. 68, and the Cruz Varas and Others v. Sweden judgment of 20 March 1991, Series A no. 201, p. 32, para. 88).
39.   In this case, therefore, the Court’s task is to determine to what extent it is true that Ersin’s move to Switzerland would be the only way for Mr Gül to develop family life with his son.
40.   The applicant left Turkey in 1983 and made his way to Switzerland, where he applied for political asylum; this application was rejected by the Minister for Refugees in 1989 (see paragraph 10 above).  His wife joined him in 1987 so that she could receive medical treatment in Switzerland after a serious accident.  Their daughter Nursal was placed from birth in a home in Switzerland and has remained there ever since (see paragraph 9 above).  In 1990 Mr and Mrs Gül were granted a residence permit on humanitarian grounds and then sought permission to bring their son Ersin to Switzerland.  Ersin has always lived in Turkey (see paragraph 19 above).
41.   By leaving Turkey in 1983, Mr Gül caused the separation from his son, and he was unable to prove to the Swiss authorities - who refused to grant him political refugee status - that he personally had been a victim of persecution in his home country. In any event, whatever the applicant’s initial reasons for applying for political asylum, the visits he has made to his son in recent years tend to show that they are no longer valid.  His counsel, moreover, expressly confirmed this at the hearing.  In addition, according to the Government, by virtue of a social security convention concluded on 1 May 1969 between Switzerland and Turkey, the applicant could continue to receive his ordinary invalidity pension and half of the supplementary benefit he receives at present in respect of his wife, his son Ersin and his daughter Nursal if he returned to his home country (see paragraph 23 above).
Mrs Gül’s return to Turkey is more problematic, since it was essentially her state of health that led the Swiss authorities to issue a residence permit on humanitarian grounds.  However, although her state of health seemed particularly alarming in 1987, when her accident occurred, it has not been proved that she could not later have received appropriate medical treatment in specialist hospitals in Turkey.  She was, moreover, able to visit Turkey with her husband in July and August 1995 (see paragraph 19 above).
Furthermore, although Mr and Mrs Gül are lawfully resident in Switzerland, they do not have a permanent right of abode, as they do not have a settlement permit but merely a residence permit on humanitarian grounds, which could be withdrawn, and which under Swiss law does not give them a right to family reunion (see paragraph 18 above).
42.   In view of the length of time Mr and Mrs Gül have lived in Switzerland, it would admittedly not be easy for them to return to Turkey, but there are, strictly speaking, no obstacles preventing them from developing family life in Turkey.  That possibility is all the more real because Ersin has always lived there and has therefore grown up in the cultural and linguistic environment of his country.  On that point the situation is not the same as in the Berrehab case, where the daughter of a Moroccan applicant had been born in the Netherlands and spent all her life there (see the Berrehab judgment previously cited, p. 8, para. 7).
43.   Having regard to all these considerations, and while acknowledging that the Gül family’s situation is very difficult from the human point of view, the Court finds that Switzerland has not failed to fulfil the obligations arising under Article 8 para. 1 (art. 8-1), and there has therefore been no interference in the applicant’s family life within the meaning of that Article (art. 8-1).
FOR THESE REASONS, THE COURT
Holds by seven votes to two that there has been no breach of Article 8 (art. 8) of the Convention.
Done in English and in French, and delivered at a public hearing in the Human Rights Building, Strasbourg, on 19 February 1996.

		Rudolf BERNHARDT
		President

Herbert PETZOLD
Registrar
In accordance with Article 51 para. 2 (art. 51-2) of the Convention and Rule 53 para. 2 of Rules of Court A, the dissenting opinion of Mr Martens, approved by Mr Russo, is annexed to this judgment.
R. B.
H. P.


DISSENTING OPINION OF JUDGE MARTENS, APPROVED BY JUDGE RUSSO
A. Introduction
1.   To my regret I have not been able to persuade the majority. I remain unable to share their opinion.  I will refrain from arguing why, but just set out my own judgment.  I trust that from that judgment it will be sufficiently clear why I could not join the majority.
2.   What is at stake in this case is whether the refusal of the Swiss authorities to grant the applicant’s son Ersin authorisation to reside in Switzerland with his parents violated Switzerland’s obligation under Article 8 (art. 8) to respect the applicant’s family life.  Consequently, the circumstances obtaining at the date of the (first) refusal of the requested authorisation - that is 19 September 1990 - are decisive.
I will come back to these circumstances hereinafter (see paragraph 14), but I note already here that on 19 September 1990 the applicant and his wife were living lawfully in Switzerland having been granted a residence permit on humanitarian grounds on 15 February 1990.  Their son Ersin, who was born on 20 January 1983, was then 7 years old and lived in Turkey under circumstances which still remain controversial (see paragraph 12 below).
3.   One more preliminary remark with regard to the facts.  The Court has repeatedly stressed that it is not bound by the Commission’s findings of fact and remains free to make its own appreciation in the light of all the material before it (see, inter alia, the Cruz Varas and Others v. Sweden judgment of 20 March 1991, Series A no. 201, p. 29, para. 74).  In doing so we should, however, bear in mind our limitations and be particularly careful not to take into account facts other than those which are properly established.  The Government have contended - without basing their contention on specific facts - that in 1983 the applicant left Turkey "of his own free will, preferring to seek employment in Switzerland" thereby suggesting that the applicant’s assertion that he came to Switzerland as a refugee was a falsehood.  However, although the applicant sought asylum on 26 April 1983, his application was only dismissed on 9 February 1989 together with that of his wife (which dated from 8 February 1988).  The applicant appealed.  This appeal was never decided because the applicant withdrew his application, since - as his counsel put it without being contradicted - pursuing the application for asylum was incompatible with accepting the residence permit on humanitarian grounds that had been offered to him and his wife.  Under these circumstances it is not for us to simply base ourselves on the refusal at first instance or to speculate, thirteen years hence, on the truth and relevance of the assertions underlying the applicant’s asylum request.  True, it is common ground that in the summer of 1995 the applicant visited Ersin in Turkey and, although attracting notice by a press interview, has apparently not experienced any disagreeableness from the Turkish authorities.  However, that does not in itself warrant the conclusion that thirteen years earlier, in 1983, the applicant had no relevant and sufficient grounds for fleeing from persecution in Turkey and requesting asylum in Switzerland.
B. Applicability of Article 8 (art. 8)
4.   In its Abdulaziz, Cabales and Balkandali v. the United Kingdom judgment of 28 May 1985, Series A no. 94, the Court adopted the established doctrine of the Commission that although, certainly, the right of aliens to enter or to remain in a country is not as such guaranteed by the Convention, immigration controls have to be exercised consistently with Convention obligations and that, accordingly, the exclusion of a person from a State where members of his family are living may raise an issue under Article 8 (art. 8) (see paragraphs 59 and 60 of the judgment). Since this judgment there has been a considerable evolution in the Court’s general doctrine on Article 8 (art. 8), but not on this point.  On the contrary, its subsequent case-law has solidly confirmed the principle that, although Contracting States have, as a matter of well-established international law, the right to control the entry, residence and expulsion of aliens, that right is subject to their obligations under the Convention, notably those under Article 8 (art. 8) (see the Berrehab v. the Netherlands judgment of 21 June 1988, Series A no. 138, pp. 15-16, paras. 28-29; the Moustaquim v. Belgium judgment of 18 February 1991, Series A no. 193, p. 19, para. 43; the Cruz Varas and Others judgment cited above, p. 28, para. 70; the Vilvarajah and Others v. the United Kingdom judgment of 30 October 1991, Series A no. 215, p. 34, para. 102; the Beldjoudi v. France judgment of 26 March 1992, Series A no. 234-A, p. 27, para. 74; and the Nasri v. France judgment of 13 July 1995, Series A no. 320-B, p. 25, para. 41).
Accordingly, if on 19 September 1990 there existed "family life" between the applicant and his son Ersin, the applicability of Article 8 (art. 8) to the facts of the present case cannot be called into question.  To the evolution in the Court’s general doctrine on Article 8 (art. 8) I will return in paragraph 7 below.
5.   On 19 September 1990 there certainly was a family life relationship between the applicant and Ersin.  Since Ersin was born from the legitimate marriage between the applicant and his wife, it follows from the aforementioned Berrehab judgment (p. 14, para. 21) that there is ipso facto such a relationship (see also the Hokkanen v. Finland judgment of 23 September 1994, Series A no. 299-A, p. 19, para. 54).  True, as the Court recognised in the Berrehab judgment, subsequent events may break such a family life relationship, but only exceptional circumstances can warrant the conclusion that the tie between a parent and his or her child is severed.  The mere fact that, at the relevant date, the applicant had not seen his then seven-year-old son for almost seven years is not sufficient to produce this negative effect.  In this context it is immaterial whether the applicant left his wife and Ersin under fear from political prosecution or purely for economic reasons.
C. Is Switzerland in breach of an obligation under Article 8 (art. 8)?
6.   "According to the Court’s well established case-law, ‘the mutual enjoyment by parent and child of each other’s company constitutes a fundamental element of family life’", as the Court pointed out in paragraph 86 of the McMichael v. the United Kingdom judgment of 24 February 1995, Series A no. 307-B, p. 55. Consequently, decisions of State authorities hindering such enjoyment in principle amount to an infringement of the State’s obligation to respect the family life of those concerned.  It follows that the refusal of the Swiss authorities to grant the applicant’s son Ersin authorisation to reside in Switzerland in principle entails their responsibility under Article 8 (art. 8).
Before it is possible to assess whether the refusal was justified, it is - alas - necessary to give some consideration to the question whether or not Switzerland’s obligation under Article 8 (art. 8) is a positive or a negative one.
D. Positive or negative obligation?
7.   The Court’s case-law distinguishes between positive and negative obligations.  Negative obligations require member States to refrain from action, positive to take action.  The Court has repeatedly stressed that the boundaries between the two types "do not lend themselves to precise definition" (see, for instance, the Keegan v. Ireland judgment of 26 May 1994, Series A no. 290, p. 19, para. 49).  The present case well illustrates the truth of this proposition since the question whether the Swiss decision violated a positive or a negative obligation, if either, seems hardly more than one of semantics: the refusal of the Swiss authorities to let Ersin and his parents be reunited may be considered as an action from which they should have refrained, whereas it could arguably also be viewed as failing to take an action which they were required to take, namely making a reunion possible by granting the authorisation.  If one takes the view that, if there is a violation at all, it must be of a positive obligation - a view that finds support in the aforementioned Abdulaziz, Cabales and Balkandali judgment (see paragraph 4 above) - then one has to put up with the rather awkward systematic inconsistency that exclusion of a person from a state where his family lives does not fall into the same category of breaches as expulsion of a person from a state where his family lives: the former decision may be in breach of a positive obligation under Article 8 (art. 8), whereas the latter may be in breach of a negative obligation.
8.   These and other difficulties in distinguishing between cases where positive and cases where negative obligations are at stake would be immaterial if both kinds of obligation were treated alike.  There was a time, however, when the Court’s case-law did treat them differently.
The Abdulaziz, Cabales and Balkandali judgment is a striking instance: see paragraph 67 of that judgment.  Under the pretext of the vagueness of the notion "respect" in Article 8 (art. 8) the Court held that its requirements will vary from case to case, thus creating for itself the possibility of taking into account, when establishing whether or not there is a positive obligation, whether or not there is a consensus between member States and, moreover, a wide margin of appreciation for the State concerned. This approach has been rightly criticised both outside and inside the Court.  One of the main objections was that under this doctrine, in the context of positive obligations, the margin of appreciation might already come into play at the stage of determining the existence of the obligation, whilst in the context of negative obligations it only plays a role, if at all, at the stage of determining whether a breach of the obligation is justified.
The Court’s doctrine on this point has, however, evolved considerably since the Abdulaziz, Cabales and Balkandali judgment.  The aforementioned difference in treatment between positive and negative obligations has gradually dwindled away. The Court now holds that the applicable principles are similar, adding that in both contexts regard must be had to the fair balance that has to be struck between the competing interests of the individual and the community (see, inter alia: the above-mentioned Keegan judgment, loc. cit. (paragraph 7 above); the above-mentioned Hokkanen judgment, p. 20, para. 55; and the Stjerna v. Finland judgment of 25 November 1994, Series A no. 299-B, p. 61, para. 39).
9.   For present purposes it may, therefore, be assumed that it makes no material difference whether a positive or a negative obligation is at stake.  The present doctrine notably implies that the distinction between the two types of obligation has no bearing on either the burden of proof or the standards for assessing whether a fair balance has been struck.
It follows that the refusal of the Swiss authorities to grant the applicant’s son Ersin authorisation to reside in Switzerland amounts to a violation of Article 8 (art. 8), unless it is deemed justified under paragraph 2 of that Article (art. 8-2) or under similar principles to those enshrined therein.
I agree with the Commission that the requirements of "in accordance with the law" and "legitimate aim" are fulfilled.  The applicant’s argument that the refusal was not "in accordance with the law" did not convince me.  On the other hand I cannot help saying that I consider the Government’s attempt to embellish the harsh, political objectives of their decision by pleading that in the first place it was designed to serve Ersin’s interests rather hypocritical.  The stress laid on financial considerations makes it clear that the legitimate aim pursued was, if not only then mainly, "the interests of the economic well-being of the country".
It follows that in any event the decisive question is whether the refusal of authorisation to reside in Switzerland was proportionate.
E. Was the refusal proportionate?
10.   Was it "necessary in a democratic society" to refuse the applicant’s seven-year-old son Ersin authorisation to come and live in Switzerland with his parents?  In other words, did that decision of the Swiss authorities strike a fair balance between the competing interests of the applicant, his wife and their son on the one hand and those of the community as a whole on the other?
11.   In explaining the interests of the community the Government have stressed that Switzerland has a very high percentage of foreigners living within its borders.  Hence, as counsel for the Government put it at our hearing, "in Switzerland immigration is a particularly sensitive subject".  Against this background the Government are, understandably, afraid of creating a precedent and therefore emphasise - rightly - that what is at stake is their right to control the entry of non-nationals into their territory and that, accordingly, we should leave them a wide margin of appreciation.  In this context they stress that they have only granted the applicant and his wife a temporary residence permit on humanitarian grounds, that as a consequence of that generosity they have already to bear the costs of subsistence of the applicant, his wife and their daughter Nursal and that it is therefore asking too much to expect them to do the same for Ersin.
12.   So much for the one scale of the balance.  What lies in the other?  First and foremost, of course, a fundamental element of an elementary human right, the right to care for your own children.  It was only natural that the applicant and his wife, as soon as their residence situation was regularised, wanted their seven-year-old son to live with them.  There is a dispute as to Ersin’s living conditions, but I need not go deeply into that.  It suffices to note that the Government have not convincingly established that those conditions were satisfactory, let alone that, at the decisive moment, it was more in the interest of Ersin to remain in Turkey than to be reunited with his father and mother.
13.   The Government do not argue that these are not weighty interests.  But they seek to diminish their relevance by contending that the applicant - on whom, they add, is the burden - has not shown that there are obstacles to re-establishing the family - father, mother and Ersin - in Turkey.  It is clear that the Government are thus relying on paragraph 68 of the Abdulaziz, Cabales and Balkandali judgment.  However, they choose to ignore the fact that the Court, in the first sentence of that paragraph, explicitly distinguishes "the present proceedings" - i.e. the cases of the three wives that were before the Court - from the case of "immigrants who already had a family which they left behind in another country until they had achieved settled status in the United Kingdom" (= the country of settlement).
That is an important proviso, for it strongly suggests that in a case of "immigrants who already had a family which they left behind" - such as the present applicant - different norms should be applied.
14.   Which norms?  The Court does not answer that question, but it is natural to infer that it intended to make it clear that in respect of such cases it might possibly hold that, in the context of the issue of family reunion, the State of settlement should respect the choice of the immigrants who have achieved settled status there and, accordingly, must accept members of their family which they had left behind for settlement.
In other words, contrary to the Government’s suggestion, the Abdulaziz, Cabales and Balkandali judgment is no authority for their allegation that Switzerland may refuse Ersin entry - although he is a member of the family which the applicant and his wife left behind - on the mere ground that if the applicant and his wife want family reunion they should go back to Turkey, there being a violation of Switzerland’s obligations under Article 8 (art. 8) only if the applicant proves that there are obstacles to doing so or other special reasons why that could not be expected of him.
On the contrary, the Abdulaziz, Cabales and Balkandali judgment supports the proposition that in cases where a father and mother have achieved settled status in a country and want to be reunited with their child which for the time being they have left behind in their country of origin, it is per se unreasonable, if not inhumane to give them the choice between giving up the position which they have acquired in the country of settlement or to renounce the mutual enjoyment by parent and child of each other’s company which constitutes a fundamental element of family life.
15.   It remains, of course, to be considered whether the latter principle applies in the present case, where the applicant has not "achieved settled status" in Switzerland, in so far as he and his wife have not been granted a "settlement permit", but have to base their right of residence on a permit which has, in principle, a temporary character and, consequently, a lower legal status than a settlement permit.
It cannot be denied that, from a point of view of State interest - that is from a point of view of immigration and residence - there is a good case for answering this question in the negative.  However, the European Court of Human Rights has to ensure, in particular, that State interests do not crush those of an individual, especially in situations where political pressure - such as the growing dislike of immigrants in most member States - may inspire State authorities to harsh decisions. As we stressed in paragraph 29 of our aforementioned Berrehab judgment (see paragraph 4 above), the Court must examine cases like this not only from the point of view of immigration and residence, but also with regard to the mutual interests of the applicant, his wife and Ersin.
Whether he came as a refugee (as we must presume (see paragraph 3 above)) or as a job seeker (as the Government allege), at the material time the applicant had been living in Switzerland for seven years and his wife for four years.  During these years he had been legally employed, apparently by the same employer, until an unspecified date in 1990 when he fell ill (see paragraph 7 of the Court’s judgment).  The Swiss authorities have taken this time element into consideration, since their decision to grant a residence permit was partly based on the time the applicant had been living in Switzerland (see paragraph 11 of the Court’s judgment).  Rightly so, for, generally speaking, it may be assumed that after a period of between three and five years immigrants become rooted in the country of settlement.  By then they have formed new social ties there and have definitively begun to adapt themselves to their new homeland.  In assessing the humaneness of the choice with which the Swiss authorities confronted the applicant and his wife this element, the fact that they have become integrated in their new homeland - an element which, incidentally, is closely connected with their private life - is of far more importance than the formal status of their permit.
There are some further, specific elements to be taken into account.
The first is that for the applicant and his wife the choice in question was not only between renouncing their son or renouncing the position which they had acquired in Switzerland, but also between renouncing their son Ersin or their little daughter Nursal who was being educated in a home in Switzerland and whose interests almost certainly would have required that she should be left behind.
The second is that the applicant’s wife is dependent on medical care which she can certainly get in Switzerland, whilst it is in debate to what extent, if at all, she will be able to get it in Turkey.
The third is that the mere fact that the Turkish authorities did not immediately arrest the applicant when he entered the country as a visitor does not imply that he would not get into trouble if he tried to settle there again on a permanent basis.
The fourth is that the applicant and his wife deserve compassion: whilst his wife had been suffering from epilepsy since 1982 and had a terrible accident in 1987, the applicant himself became disabled in 1990.
Under these circumstances it could not reasonably be required of the applicant and his wife that in order to be reunited with Ersin they should leave Switzerland and return to Turkey.
It follows that a proper balance was not achieved between the interests involved, that the refusal of the Swiss authorities is disproportionate and, as such, not necessary in a democratic society. I thus conclude that there was a violation of Article 8 (art. 8).
"""

In [57]:
text_41 = """THE FACTS

      The applicant is an Austrian citizen who lives in Bregenz.  He is represented before the Commission by Mr. L. W. Weh, a lawyer
practising in Bregenz.

      The facts of the case, as submitted by the parties, may be summarised as follows:

      On 11 June 1987 the applicant was fined by a penal order (Straferkenntnis) AS 9,000 with provision for 360 hours' detention in default for failure to submit to a breath test, contrary to Section 99 (1) (b) of the Road Traffic Act 1960 (Straßenverkehrsordnung).  He appealed to the Regional Government (Landesregierung) which, on 11
November 1987, rejected his appeal.

      On 23 March 1988 the Administrative Court
(Verwaltungsgerichtshof) quashed the decision of the Regional
Government of 11 November 1987 and remitted the case to that authority.
The Regional Government's second decision, of 23 December 1988, reduced the penalty from AS 9,000 to AS 5,000 and the period of imprisonment in default from 360 hours to 200 hours.

      The applicant's complaint to the Constitutional Court (Ver-
fassungsgerichtshof) was rejected on 10 March 1989 on the ground that it had no sufficient prospect of success and that the case was not outside the competence of the Administrative Court.  The Constitutional
Court referred principally to its own case-law on Article 6 of the
Convention in finding that the application had no sufficient prospect of success.

      On 10 November 1989 the Administrative Court gave its second decision in the case.  It found that it was not prevented from considering that the factual position had been determined in a relevant and conclusive way, although it was not able to review whether the defence's version of the facts was correct.  Accordingly, the
Administrative Court could not decide whether the applicant had or had not spoken of a "good session" (drinking).  As to the applicant's complaint that his lawyer had not been able to examine a witness, the court noted that an oral hearing was not a necessary part of the administrative criminal proceedings.  As to the alleged unconstitutionality of the Austrian reservation to Article 5 of the Convention, the Court referred to previous case-law.  The complaint was dismissed as a whole.


COMPLAINTS

      The applicant alleges a violation of Article 6 of the Convention in that the administrative criminal proceedings brought against him were determined initially by the administrative authorities which did not constitute independent and impartial tribunals within the meaning of Article 6 para. 1 of the Convention, and subsequently by the
Constitutional Court and the Administrative Court, the scope of whose review is not sufficient to comply with Article 6 of the Convention, and which cannot decide the case themselves.

He also makes specific complaints about the nature of the administrative authorities' examination of the case, including his inability to put questions to prosecution witnesses, and about the inevitably partial status of experts.



PROCEEDINGS BEFORE THE COMMISSION

      The application was introduced on 13 June 1990 and registered on
10 July 1990.

      On 16 October 1991 the Commission decided to request the parties to submit their written observations on the admissibility and merits of the application.

      The respondent Government submitted their observations on 21 February 1992 and the applicant submitted his observations on 5 October 1992.

      On 15 February 1993 the Commission decided to hear the parties as to the admissibility and merits of this case and Applications Nos. 15523/89, 15527/89, 15963/90, 16713/90 and 16718/90.  At the hearing the parties were represented as follows:

For the Government:

Ambassador F. Cede          Legal Adviser, Federal Ministry for Foreign
                            Affairs, Agent

Ms. S. Bernegger            Federal Chancellery, Adviser

For the applicant:

Mr. W.L. Weh                Representative



THE LAW

      The applicant alleges violation of Article 6 (Art. 6) of the
Convention.

      The Government submit that the Austrian reservation to Article
5 (Art. 5) of the Convention prevents the Commission from examining the case.  They accept, however, that if the reservation does not prevent an examination of the case, then the review of administrative decisions by the Administrative Court and Constitutional Court was not sufficiently wide to comply with Article 6 para. 1 (Art. 6-1) of the
Convention.  They add, in this respect, that although the offence for which the applicant was convicted under Section 99 (1)(b) of the Road Traffic Act 1960 (refusing to take a breath test) was not, as such, in force on the date of the reservation,  the law then in force did impose an obligation on road users to drive with reasonable consideration for other road users and to pay such attention as is required for the maintenance of order, safety and traffic efficiency.


      The Government consider that the absence of an oral public and direct hearing is covered by the Austrian reservation to Article 6 (Art. 6) of the Convention.  They also point out that the applicant did not make a complaint about the absence of a hearing before the Administrative Court.

      The applicant considers that the Austrian reservation to Article 5 (Art. 5) of the Convention is neither valid nor applicable in the present case.  He agrees that the scope of review by the Constitutional Court and Administrative Court does not comply with Article 6 (Art. 6) of the Convention. He considers that the reservation to Article 6 (Art. 6), if valid, is not applicable to the present proceedings.

      In connection with Article 144 para. 2 of the Federal Constitution, the Government consider that, although that provision provides for non-acceptance of a constitutional complaint on grounds which were not in force in 1958 when the reservation was made, the possibility for the Constitutional Court to refuse to deal with appeals against decisions without giving detailed reasons is only a procedural limitation and not a substantive one.  They point out that any appeal lodged with the Constitutional Court against a decision is subject to comprehensive review.

      The applicant in this respect considers that the limitation of the Constitutional Court's jurisdiction by Article 144 para. 2 of the Federal Constitution does not meet the requirements of the reservation, even if it applies.

      The Commission finds that the application raises complex issues of law under the Convention, including questions concerning the Austrian reservations to Articles 5 and 6 (Art. 5, 6) of the Convention, the determination of which must be reserved for an examination on the merits.

      The application cannot therefore be declared manifestly ill- founded within the meaning of Article 27 para. 2 (Art. 27-2) of the Convention.  No other ground for declaring it inadmissible has been established.

      For these reasons the Commission unanimously

      DECLARES THE APPLICATION ADMISSIBLE, without prejudging the
      merits of the case.

"""

In [58]:
def extract_relevant_sections(text: str) -> List[dict]:
    """Return [{"name", "start", "end", "text"}] only for the relevant sections"""

    #To find each section
    header_alternatives = "|".join(re.escape(h) for h in SECTION_HEADERS)
    header_re = re.compile(
        rf"^[ \t]*[IVXLC0-9.]*[ \t]*({header_alternatives})[ \t]*$",
        re.MULTILINE,
    )

    #We identify the begining and the end of each title
    matches = list(header_re.finditer(text))
    dissenting_matches = list(DISSENTING_OPINION_RE.finditer(text))
    all_headers = matches + dissenting_matches

    sections = []
    for i, sec in enumerate(all_headers):
        #start after the name of the section
        content_start, name = sec.end(), sec.group(1).strip()
        if name in KEEP_SECTIONS or name.startswith("DISSENTING OPINION"):
            #finish when star the next one
            content_end = all_headers[i + 1].start() if i + 1 < len(all_headers) else len(text)
            sections.append({
                "name": name,
                "start": content_start,
                "end": content_end,
                "text": text[content_start:content_end],
            })

    if not sections:
        # If we dont find relevant section we return all the document
        print("WARNING: No known section headers were detected. The entire document will be used.")
        sections = [{"name": "FULL_DOCUMENT", "start": 0, "end": len(text), "text": text}]

    return sections

In [107]:
@lru_cache(maxsize=1)
def _get_sat_model(model_name: str = "sat-3l-sm"):
    """Loads the model once per process"""
    from wtpsplit import SaT
    return SaT(model_name)


def sentence_segments_for_section(section: str, sat_model=None) -> List[str]:
    """identify the segments in a section
    section (str): the text of the section
    return (list) with the segments"""
    sat_model = sat_model or _get_sat_model()
    sentences = sat_model.split(section)
    return sentences

In [136]:
def _cosine_sim(a: np.ndarray, b: np.ndarray) -> float:
    denom = (np.linalg.norm(a) * np.linalg.norm(b)) or 1e-8
    return float(np.dot(a, b) / denom)

def merge_until_max(segments: List[str], max_size: int) -> List[str]:
    final = []
    aux = ""

    for segment in segments:
        if len(aux) + len(segment) > max_size:
            if aux:
                final.append(aux)
            aux = segment
        else:
            aux += segment
    if aux:
        final.append(aux)

    return final

def merge_adjacent_by_similarity(sentences: List[str], max_size: int = 5000) -> List[dict]:
    """
    1. Embed all sentences.
    2. Calculate the cosine similarity between each sentence and the next one.
    3. Detect topic shifts using `find_peaks()` on the dissimilarity
    (1 - similarity), a  peak there indicates a major topic shift.
    4. Merge ADJACENT segments belonging
    """
    if not sentences or len(sentences) == 1:
        return sentences

    #embeddings = embed_texts(sentences)
    #n = len(sentences)

    #adjacent_sims = np.array([_cosine_sim(embeddings[i], embeddings[i + 1]) for i in range(n - 1)])

    #peak_idx, _ = find_peaks(1 - adjacent_sims)
    #cut_points = sorted(set(int(p) for p in peak_idx))
    #bounds = [0] + [c + 1 for c in cut_points] + [n] #add the start and the end with the cut points
    #natural_ranges = [(bounds[i], bounds[i + 1]) for i in range(len(bounds) - 1)]

    #segments = []
    #for start, end in natural_ranges:
    #    segments.append("\n".join(sentences[start:end]))

    return merge_until_max(segments, max_size)

In [130]:
def make_segmentation(text: str, max_size: int = 5000) -> List[dict]:
    """Given a document, identifies sections that might contain arguments and segments them"""
    sections = extract_relevant_sections(text)

    sat_model = _get_sat_model()
    segments = []
    for section in sections:
        text = re.sub(r"\s+", " ", section["text"]).strip() #clean the spaces in text
        if len(text.split()) > max_size:
          all_sentences = sentence_segments_for_section(text, sat_model)
          segments.extend(merge_adjacent_by_similarity(all_sentences, max_size))
        else:#if the section fit in the LLM we keep it
          segments.append(text)

    return segments

In [137]:
segments = make_segmentation(text_41)
segments

["The applicant alleges a violation of Article 6 of the Convention in that the administrative criminal proceedings brought against him were determined initially by the administrative authorities which did not constitute independent and impartial tribunals within the meaning of Article 6 para. 1 of the Convention, and subsequently by the Constitutional Court and the Administrative Court, the scope of whose review is not sufficient to comply with Article 6 of the Convention, and which cannot decide the case themselves. He also makes specific complaints about the nature of the administrative authorities' examination of the case, including his inability to put questions to prosecution witnesses, and about the inevitably partial status of experts.",
 "The applicant alleges violation of Article 6 (Art. 6) of the Convention. The Government submit that the Austrian reservation to Article 5 (Art. 5) of the Convention prevents the Commission from examining the case. They accept, however, that if

In [138]:
len(segments)

2

In [139]:
segments_14 = make_segmentation(text_14)
segments_14

['I. ALLEGED VIOLATION OF ARTICLE 8 (art. 8) OF THE CONVENTION 28. Mr Gül submitted that the Swiss authorities’ refusal to permit his son Ersin to join him in Switzerland had infringed his right to respect for his family life. He relied on Article 8 (art. 8) of the Convention, which provides: "1. Everyone has the right to respect for his ... family life ... 2. There shall be no interference by a public authority with the exercise of this right except such as is in accordance with the law and is necessary in a democratic society in the interests of national security, public safety or the economic well-being of the country, for the prevention of disorder or crime, for the protection of health or morals, or for the protection of the rights and freedoms of others." 29. It is first necessary to determine whether there is a "family life" within the meaning of Article 8 (art. 8). 30. The Government’s primary submission was that Article 8 (art. 8) was not applicable, since in the instant case 

In [140]:
len(segments_14)

7